### System Information

In [1]:
import sys
import platform
import torch
import torchvision
import numpy as np
import scipy

print("Python       :", sys.version)
print("Platform     :", platform.platform())

print("\n--- Libraries ---")
print("PyTorch      :", torch.__version__)
print("Torchvision  :", torchvision.__version__)
print("NumPy        :", np.__version__)
print("SciPy        :", scipy.__version__)

print("\n--- CUDA ---")
print("CUDA runtime :", torch.version.cuda)
print("Available    :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU          :", torch.cuda.get_device_name(0))
    print(
        "VRAM         :",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

Python       : 3.10.6 (tags/v3.10.6:9c7b4bd, Aug  1 2022, 21:53:49) [MSC v.1932 64 bit (AMD64)]
Platform     : Windows-10-10.0.26200-SP0

--- Libraries ---
PyTorch      : 2.6.0+cu126
Torchvision  : 0.21.0+cu126
NumPy        : 1.26.4
SciPy        : 1.13.1

--- CUDA ---
CUDA runtime : 12.6
Available    : True
GPU          : NVIDIA GeForce RTX 3060 Laptop GPU
VRAM         : 6.0 GB


### CUDA computation test

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

a = torch.randn(2048, 2048, device=device)
b = torch.randn(2048, 2048, device=device)

c = a @ b

print("Device :", c.device)
print("Shape  :", c.shape)
print("Mean   :", c.mean().item())

Device : cuda:0
Shape  : torch.Size([2048, 2048])
Mean   : -0.0007310984656214714


### Pycil dependency

In [3]:
import quadprog
import ot
import sklearn
import pandas
import matplotlib

print("All additional dependencies imported successfully.")

All additional dependencies imported successfully.


In [4]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)

c:\Skripsi\Project\PyCIL


In [5]:
from utils.data_manager import DataManager
from utils import factory

print("PyCIL imports: OK")

PyCIL imports: OK


In [6]:
dummy_args = {
    "dataset": "imagenet100",
    "model_name": "finetune",
    "init_cls": 50,
    "increment": 10,
}

In [7]:
from convs.resnet import resnet18

model18 = resnet18(
    pretrained=False,
    args=dummy_args
).to(device)

x = torch.randn(
    2, 3, 256, 256,
    device=device
)

with torch.no_grad():
    output = model18(x)

print("Feature shape:", output["features"].shape)

for i, fmap in enumerate(output["fmaps"], start=1):
    print(f"Layer {i}:", fmap.shape)

Feature shape: torch.Size([2, 512])
Layer 1: torch.Size([2, 64, 128, 128])
Layer 2: torch.Size([2, 128, 64, 64])
Layer 3: torch.Size([2, 256, 32, 32])
Layer 4: torch.Size([2, 512, 16, 16])


In [8]:
from convs.resnet import resnet34

model34 = resnet34(
    pretrained=False,
    args=dummy_args
).to(device)

with torch.no_grad():
    output34 = model34(x)

print("ResNet-34 feature shape:", output34["features"].shape)

ResNet-34 feature shape: torch.Size([2, 512])


In [9]:
def count_params(model):
    return sum(p.numel() for p in model.parameters())


print(
    f"ResNet-18 backbone: {count_params(model18):,}"
)

print(
    f"ResNet-34 backbone: {count_params(model34):,}"
)

ResNet-18 backbone: 11,168,832
ResNet-34 backbone: 21,276,992


In [10]:
model18.train()

x = torch.randn(
    2, 3, 256, 256,
    device=device
)

features = model18(x)["features"]

loss = (features ** 2).mean()

loss.backward()

print("Loss:", loss.item())
print("Backward pass: OK")

Loss: 0.6841769218444824
Backward pass: OK
